# Session 0 — Your turn: Teaching a machine from examples

You will redo the experiment chain from the session on the iris flowers: split, baseline, a hand-written rule, a model, and a look at its mistakes.
The split is already made for you (112 training flowers, 38 test flowers, seed 42, the same as in the session).

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np, pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
species = iris.target_names.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(len(X_train), "training flowers,", len(X_test), "test flowers")
from aij_check import check, close, summary

112 training flowers, 38 test flowers


## Task 1: Two things to beat

The baseline always answers with the most common species in the **training** set. The hand-written rule uses only the petal width:
setosa below 0.8 cm, versicolor from 0.8 up to (not including) 1.75 cm, virginica from 1.75 cm up.
Compute the baseline's accuracy on the test set, write the rule as a function that returns 0, 1 or 2 for every row of a table, and compute its accuracy on the test set.

In [2]:
majority_class = y_train.mode().iloc[0]
baseline_accuracy = accuracy_score(y_test, np.full(len(y_test), majority_class))


def width_rule(table):
    width = table["petal width (cm)"]
    return np.where(width < 0.8, 0, np.where(width < 1.75, 1, 2))


rule_accuracy = accuracy_score(y_test, width_rule(X_test))
print(baseline_accuracy, rule_accuracy)

0.3157894736842105 0.9473684210526315


In [3]:
check("the baseline is about 0.32", lambda: close(baseline_accuracy, 0.316, 0.005), "Use y_train.mode() and compare with y_test.")
check("the rule returns one answer per flower", lambda: len(width_rule(X_test)) == len(X_test))
check("the rule scores above 0.90 on the test set", lambda: rule_accuracy > 0.90, "Check the two thresholds: 0.8 and 1.75.")
check("the rule beats the baseline by a lot", lambda: rule_accuracy - baseline_accuracy > 0.5)
summary()

PASS  the baseline is about 0.32
PASS  the rule returns one answer per flower
PASS  the rule scores above 0.90 on the test set
PASS  the rule beats the baseline by a lot

4 of 4 checks passed.  Well done.


**Why it works.** The baseline says what 'doing nothing clever' scores (about one flower in three). Petal width alone separates setosa perfectly and the other two species mostly, which is why a rule written by eye gets so close to a trained model.

## Task 2: Fit a model, then read its mistakes

Fit a pipeline of `StandardScaler` and `LogisticRegression` (use `random_state=42` and `max_iter=1000`) on the training flowers.
Store its test accuracy in `model_accuracy`, and the table of test flowers it gets wrong in `wrong_flowers`, with a column `actual` and a column `predicted` holding the species names.

In [4]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)
predictions = model.predict(X_test)
model_accuracy = accuracy_score(y_test, predictions)
wrong = predictions != y_test.to_numpy()
wrong_flowers = X_test[wrong].copy()
wrong_flowers["actual"] = [species[i] for i in y_test[wrong]]
wrong_flowers["predicted"] = [species[i] for i in predictions[wrong]]
wrong_flowers

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual,predicted
134,6.1,2.6,5.6,1.4,virginica,versicolor
106,4.9,2.5,4.5,1.7,virginica,versicolor
77,6.7,3.0,5.0,1.7,versicolor,virginica


In [5]:
check("test accuracy is about 0.92", lambda: close(model_accuracy, 0.921, 0.01))
check("the model gets 3 test flowers wrong", lambda: len(wrong_flowers) == 3)
check("every mistake mixes up versicolor and virginica", lambda: set(wrong_flowers["actual"]) | set(wrong_flowers["predicted"]) <= {"versicolor", "virginica"},
      "Look at the 'actual' and 'predicted' columns: setosa should never appear.")
summary()

PASS  test accuracy is about 0.92
PASS  the model gets 3 test flowers wrong
PASS  every mistake mixes up versicolor and virginica

3 of 3 checks passed.  Well done.


**Why it works.** All three mistakes are between the two species whose petals overlap, exactly what the petal ranges in the session predicted. Reading the errors, not just the accuracy, is the last step of the chain.